<a href="https://colab.research.google.com/github/brianellis1997/Sarcasm_Detection/blob/main/DS_440_Traditional_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sarcasm Detection using NLP Techniques
### Team 2: Brian Ellis, Lindsey Rich, Elena Kern

## Load Libraries and Dataset

In [ ]:
import pandas as pd
import torch
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

#test comment

In [ ]:
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torch.optim as optim
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

In [ ]:
!git clone https://github.com/brianellis1997/Sarcasm_Detection.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
train_bal = pd.read_csv('/content/drive/MyDrive/Sarcasm_Data/Train_Balanced.csv')   # Make sure path is correct in your google drive
train_bal.head()

In [ ]:
test = pd.read_csv('/content/drive/MyDrive/Sarcasm_Data/Test_Balanced.csv')

In [ ]:
train_bal.info()

In [ ]:
train_bal['label'].value_counts()

We can see our dataset is balanced.

In [ ]:
train_bal['label'].value_counts(normalize=True)

## EDA

In [ ]:
# Separating sarcastic vs non sarcastic comments
sarcastic_com = train_bal[train_bal['label'] == 1]
non_sarcastic_com = train_bal[train_bal['label'] == 0]

In [ ]:
import nltk
from nltk import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem.porter import *
nltk.download('punkt')
nltk.download('stopwords')

import string

import gensim
from gensim.models.phrases import Phraser, Phrases
from gensim.models.word2vec import Word2Vec

import spacy # for a lemmatization example

from sklearn.manifold import TSNE

import pandas as pd
from bokeh.io import output_notebook, output_file
from bokeh.plotting import show, figure

In [ ]:
# Remove stopwords and stem to group together similar words
stpwrds = stopwords.words('english') + list(string.punctuation)
stpwrds

In [ ]:
for s in comments[:10]:
  print(s)

In [ ]:
import nltk
from nltk.corpus import stopwords

# Download stopwords if not already downloaded
nltk.download('stopwords')

# Get the English stopwords
stpwrds = set(stopwords.words('english'))

# Assuming 'comment' is the column containing the sentences in your DataFrame
comments = train_bal['comment']

# Function to tokenize, lowercase, and remove stopwords from each comment
def clean_comments(comments):
    clean_comments_list = []
    for comment in comments:
        # Tokenize and lowercase
        tokens = comment.lower().split()

        # Remove stopwords
        cleaned_tokens = [token for token in tokens if token not in stpwrds]

        clean_comments_list.append(cleaned_tokens)

    return clean_comments_list

In [ ]:
sarcastic_comments = sarcastic_com['comment']
non_sarcastic_comments = non_sarcastic_com['comment']

# Apply the function to sarcastic and non-sarcastic comments
sarcastic_clean = clean_comments(sarcastic_comments)
non_sarcastic_clean = clean_comments(non_sarcastic_comments)

# Example output for the first 10 sarcastic comments
print(sarcastic_clean[:10])

In [ ]:
from gensim.models.phrases import Phraser, Phrases

# Add common two-word phrases aka Bi-grams
sarc_bigrams = Phraser(Phrases(sarcastic_clean, min_count=32, threshold=64))
non_sarc_bigrams = Phraser(Phrases(non_sarcastic_clean, min_count=32, threshold=64))

# Apply the bigram models to the tokenized lists
clean_sarc = [sarc_bigrams[sentence] for sentence in sarcastic_clean]
clean_non_sarc = [non_sarc_bigrams[sentence] for sentence in non_sarcastic_clean]

# Example output for the first 10 sarcastic comments after adding bigrams
print(clean_sarc[:10])

In [ ]:
# WordCloud for sarcastic comments
wordcloud_sarc = WordCloud(width=800, height=400, background_color='white').generate(' '.join([' '.join(sentence) for sentence in clean_sarc]))

# Display the generated word cloud using matplotlib
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud_sarc, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud for Sarcastic Comments')
plt.show()


In [ ]:
# WordCloud for sarcastic comments
wordcloud_sarc = WordCloud(width=800, height=400, background_color='white').generate(' '.join([' '.join(sentence) for sentence in clean_non_sarc]))

# Display the generated word cloud using matplotlib
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud_sarc, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud for Non-Sarcastic Comments')
plt.show()

# Visualizing Sarcastic Comments by Subreddit

In [ ]:
subreddit_count = sarcastic_com['subreddit'].value_counts()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
top = subreddit_count.head(20)
plt.figure(figsize=(10, 6))
sns.barplot(x=top.values, y=top.index)
plt.xlabel('Number of Sarcastic Comments')
plt.ylabel('Subreddit')
plt.title('Frequency of Sarcastic Comments by Subreddit')
plt.show()


## Train Logistic Regression model (baseline)

In [ ]:
from sklearn.model_selection import train_test_split
comments = train_bal['comment']
# Create a new column 'merged_comment' by concatenating 'comment' and 'parent_comment'
train_bal['merged_comment'] = train_bal['comment'] + ' ' + train_bal['parent_comment']

labels = train_bal['label']

X_train, X_test, y_train, y_test = train_test_split(train_bal['merged_comment'], labels, test_size=0.3, random_state = 42)

print(X_train.shape)
print(X_test.shape)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# build bigrams
# and minimal word frequency
tf_idf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, lowercase=False)
# multinomial logistic regression a.k.a softmax classifier
logit = LogisticRegression(C=1, n_jobs=-1, solver='lbfgs',
                           random_state=42, verbose=1)
# sklearn's pipeline
tfidf_logit_pipeline = Pipeline([('tf_idf', tf_idf),
                                 ('logit', logit)])

In [ ]:
%%time
tfidf_logit_pipeline.fit(X_train, y_train)

In [ ]:
%%time
valid_pred = tfidf_logit_pipeline.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, valid_pred)

## Stacking with parent comment

In [ ]:
from scipy.sparse import hstack
tf_idf_comment = TfidfVectorizer(ngram_range=(1, 2), min_df=2, lowercase=False)
tf_idf_parent_comment = TfidfVectorizer(ngram_range=(1, 2), min_df=2, lowercase=False)

comment_vectors = tf_idf_comment.fit_transform(train_bal['comment'])
parent_comment_vectors = tf_idf_parent_comment.fit_transform(train_bal['parent_comment'])

# Concatenate the TF-IDF vectors horizontally
combined_vectors = hstack([comment_vectors, parent_comment_vectors])

In [ ]:
# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(combined_vectors, labels, test_size=0.2, random_state=22)

logit = LogisticRegression(C=1, n_jobs=-1, solver='lbfgs',
                           random_state=42, verbose=1)

logit.fit(X_train, y_train)
valid_pred = logit.predict(X_test)

from sklearn.metrics import accuracy_score

accuracy_score(y_test, valid_pred)

## Using non-text covariates

In [ ]:
train_bal.info()

### Author

In [ ]:
train_bal['author'].nunique()

In [ ]:
train_bal['subreddit'].nunique()

In [ ]:
train_bal.head()

In [ ]:
# Sarcastic authors
train_bal[train_bal['label'] == 1].groupby('author').size().sort_values(ascending=False).head(5)

In [ ]:
# Least sarcastic authors
train_bal[train_bal['label'] == 0].groupby('author').size().sort_values(ascending=False).head(5)

It seems that the authors with the most sarcastic comments are also the ones with the most non-sarcastic comments. Maybe author isn't a valuable predictor for sarcasm. We can test this by performing a chi-squared test.

In [ ]:
from scipy.stats import chi2_contingency

# Create a contingency table
contingency_table = pd.crosstab(train_bal['author'], train_bal['label'])

# Perform chi-square test
chi2, p, _, _ = chi2_contingency(contingency_table)
print(f"Chi-square value: {chi2}\nP-value: {p}")

With our statistical test, we see that there is very little correlation between these variables, this means that the author variable might add unwanted noise to our model and might not be worth including.

### Subreddit

In [ ]:
# Sarcastic subreddits
train_bal[train_bal['label'] == 1].groupby('subreddit').size().sort_values(ascending=False).head(5)

In [ ]:
# Non-Sarcastic subreddits
train_bal[train_bal['label'] == 0].groupby('subreddit').size().sort_values(ascending=False).head(5)

In [ ]:
train_bal['subreddit'].nunique()

In [ ]:
top_subreddits = train_bal['subreddit'].value_counts(ascending=False).head(10).index
subreddit_filter = train_bal[train_bal['subreddit'].isin(top_subreddits)]

subreddit_filter.info()

In [ ]:
sns.countplot(x='subreddit', hue='label', data=subreddit_filter)
plt.title('Sarcasm by Subreddit')
plt.xlabel('Subreddits')
plt.ylabel('Count')
plt.legend(labels=['No Sarcasm', 'Sarcasm'])
plt.xticks(rotation=90)
plt.show()

We get a promising plot that our covariate `subreddit` may have some influence on our sarcasm label. Let's test this with a chi-squared test.

In [ ]:
# Create a contingency table
contingency_table = pd.crosstab(train_bal['subreddit'], train_bal['label'])

# Perform chi-square test
chi2, p, _, _ = chi2_contingency(contingency_table)
print(f"Chi-square value: {chi2}\nP-value: {p}")

We get a statistically significant result with a p-value of 0. Let's test this with our reduced subreddit dataset.

In [ ]:
# Create a contingency table
contingency_table = pd.crosstab(subreddit_filter['subreddit'], subreddit_filter['label'])

# Perform chi-square test
chi2, p, _, _ = chi2_contingency(contingency_table)
print(f"Chi-square value: {chi2}\nP-value: {p}")

Still a significant value, this means `subreddit` is a valuable feature for predicting sarcasm.

### Up and Down Votes

In [ ]:
# Filter dataset
Votes = train_bal[['score', 'ups', 'downs', 'label']]
Votes.info()

In [ ]:
from scipy.stats import pearsonr

voting_corr = Votes.corr()

# Create an empty DataFrame to store p-values
p_values = pd.DataFrame(index=voting_corr.index, columns=voting_corr.columns)

# Loop through pairs of variables and calculate correlation and p-value
for col1 in voting_corr.columns:
    for col2 in voting_corr.index:
        corr, p_value = pearsonr(Votes[col1], Votes[col2])
        p_values.loc[col1, col2] = p_value

In [ ]:
train_bal.describe()

In [ ]:
train_bal['ups'].median()

In [ ]:
voting_corr

In [ ]:
p_values

In [ ]:
plt.hist(Votes['downs'])
plt.title('Down Vote Distribution')
plt.show()

In [ ]:
Votes['ups'].nunique()

In [ ]:
Votes['ups'].max()

In [ ]:
import numpy as np

plt.hist(Votes['ups'])
plt.title('Up Vote Distribution')
plt.show()

In [ ]:
plt.hist(Votes['score'])
plt.title('Distribution of Score')
plt.show()

In [ ]:
train_bal.info()

Maybe train a model only with voting variables to see influence?

### Date

In [ ]:
train_bal['date'].head()

In [ ]:
train_bal['date'] = pd.to_datetime(train_bal['date'])
train_bal['date'].head()

In [ ]:
train_bal['created_utc'].head()

`date` and `created_utc` may hold significance in our sarcastic prediction task. Maybe sarcastic comments are more prevelant during certain times of the year and maybe more prevelant during times of the day as well.

#### Month

In [ ]:
monthly_counts = train_bal.groupby(['label', train_bal['date'].dt.month])['date'].count().unstack()
monthly_counts

In [ ]:
monthly_counts.columns

In [ ]:
# Melt the DataFrame to have a 'month' column
monthly_counts_melted = monthly_counts.reset_index().melt(id_vars=['label'], var_name='month', value_name='count')

# Plot the counts using a bar plot
sns.barplot(x='month', y='count', hue='label', data=monthly_counts_melted)
plt.title('Counts of Each Label for Each Month')
plt.xlabel('Month')
plt.ylabel('Count')
plt.show()

It seems that there are more sarcastic comments at the beginning of the year and more non-sarcastic comments towards the end of the year.<br>We can perform a statistical test to see if there's a relationship between month and sarcasm label.

In [ ]:
from scipy.stats import chi2_contingency
# Create a contingency table
contingency_table = monthly_counts_melted.pivot_table(values='count', index='month', columns='label', fill_value=0)

contingency_table

In [ ]:
# Perform chi-square test
chi2, p_value, _, _ = chi2_contingency(contingency_table)

# Print the result
print(f"Chi-square value: {chi2}")
print(f"P-value: {p_value}")

There is a statistically significant relationship between months and sarcasm counts. <br>This tells us that including the `month` variable into our dataframe might be good for prediction.

#### Year

In [ ]:
yearly_counts = train_bal.groupby(['label', train_bal['date'].dt.year])['date'].count().unstack()
yearly_counts

In [ ]:
# Melt the DataFrame to have a 'month' column
yearly_counts_melted = yearly_counts.reset_index().melt(id_vars=['label'], var_name='year', value_name='count')

# Plot the counts using a bar plot
sns.barplot(x='year', y='count', hue='label', data=yearly_counts_melted)
plt.title('Counts of Each Label for Each Year')
plt.xlabel('Year')
plt.ylabel('Count')
plt.show()

In [ ]:
# Create a contingency table
contingency_table = yearly_counts_melted.pivot_table(values='count', index='year', columns='label', fill_value=0)

contingency_table

In [ ]:
# Perform chi-square test
chi2, p_value, _, _ = chi2_contingency(contingency_table)

# Print the result
print(f"Chi-square value: {chi2}")
print(f"P-value: {p_value}")

I don't know if `year` would have the same predictive power as `month` because it is not cyclic and has much less data.

### UTC

In [ ]:
train_bal['created_utc'] = pd.to_datetime(train_bal['created_utc'])
train_bal.info()

In [ ]:
train_bal['hour'] = train_bal['created_utc'].dt.hour

In [ ]:
sns.countplot(x='hour', hue='label', data=train_bal)
plt.title('Distribution of Sarcastic Comments by Hour of the Day')
plt.xlabel('Hour of the Day')
plt.ylabel('Count')
plt.show()

Interestingly enough, it seems like there are more sarcastic comments in the evening, around the time people come back from work/school. There are less sarcastic comments in the morning.

In [ ]:
train_bal['month'] = train_bal['date'].dt.month
train_bal['year'] = train_bal['date'].dt.year
train_bal.info()

In [ ]:
# Group by hour, month_name, and label, and calculate the count
grouped_time = train_bal.groupby(['hour', 'month', 'label']).size().reset_index(name='count')

grouped_time